# UK Road Safety Analytics
## Notebook 03: Feature Engineering

**Author:** David  
**Date:** July 2026  
**Stage:** Feature Engineering and Model Preparation  

### Objective
Create new variables from the existing data that give the machine 
learning model better signals to learn from. Then prepare the final 
training and test datasets ready for modelling in Notebook 04.

### Target Variable
We are building a binary classification model:
- **1 = Fatal or Serious** — the accident resulted in death or serious injury
- **0 = Slight** — the accident resulted in a minor injury

This framing answers the most important question: given what we know 
about an accident at the moment it happens, can we predict whether it 
will be fatal or serious?

### Features We Will Create
- Is rush hour (7-9am or 4-7pm)
- Is weekend
- Is dark (light conditions)
- Is wet or adverse road surface
- Is high speed limit (60mph or above)
- Is rural road
- Hour of day (already extracted)
- Month (already extracted)

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
import os

# Load clean dataset
df = pd.read_csv('../data/processed/road_safety_clean.csv', low_memory=False)
df['date'] = pd.to_datetime(df['date'])
df['collision_year'] = df['date'].dt.year

print(f"Dataset loaded: {df.shape}")
print(f"\nTarget variable distribution:")
print(df['severity_label'].value_counts())
print(f"\nAs percentage:")
print((df['severity_label'].value_counts(normalize=True) * 100).round(1))

Dataset loaded: (640449, 36)

Target variable distribution:
severity_label
Slight     479049
Serious    149011
Fatal       12389
Name: count, dtype: int64

As percentage:
severity_label
Slight     74.8
Serious    23.3
Fatal       1.9
Name: proportion, dtype: float64


In [2]:
# ── CREATE ALL FEATURES ───────────────────────────────────────────────────

# ── TARGET VARIABLE ───────────────────────────────────────────────────────
# 1 = Fatal or Serious, 0 = Slight
df['target'] = (df['severity_label'].isin(['Fatal', 'Serious'])).astype(int)

print(f"Target distribution:")
print(f"  Fatal or Serious (1): {df['target'].sum():,} ({df['target'].mean()*100:.1f}%)")
print(f"  Slight (0):           {(df['target']==0).sum():,} ({(1-df['target'].mean())*100:.1f}%)")

# ── TIME FEATURES ─────────────────────────────────────────────────────────
df['is_rush_hour'] = (
    ((df['hour'] >= 7) & (df['hour'] <= 9)) |
    ((df['hour'] >= 16) & (df['hour'] <= 19))
).astype(int)

df['is_weekend'] = df['day_of_week'].isin([1, 7]).astype(int)

df['is_night'] = (
    (df['hour'] >= 22) | (df['hour'] <= 5)
).astype(int)

# ── ROAD CONDITION FEATURES ───────────────────────────────────────────────
df['is_dark'] = df['light_conditions'].isin([4, 5, 6, 7]).astype(int)

df['is_adverse_surface'] = df['road_surface_conditions'].isin(
    [2, 3, 4, 5, 6, 7]).astype(int)

df['is_adverse_weather'] = df['weather_conditions'].isin(
    [2, 3, 4, 5, 6, 7]).astype(int)

# ── ROAD TYPE FEATURES ────────────────────────────────────────────────────
df['is_high_speed'] = (df['speed_limit'] >= 60).astype(int)

df['is_rural'] = (df['urban_or_rural_area'] == 2).astype(int)

df['is_single_carriageway'] = (df['road_type'] == 6).astype(int)

# ── ENCODE CATEGORICAL VARIABLES ──────────────────────────────────────────
# Speed limit as numeric — already numeric, just clean it
df['speed_limit_clean'] = pd.to_numeric(df['speed_limit'], errors='coerce')
df['speed_limit_clean'] = df['speed_limit_clean'].fillna(
    df['speed_limit_clean'].median())

print(f"\nFeatures created successfully.")
print(f"\nFeature summary:")
features_created = ['target', 'is_rush_hour', 'is_weekend', 'is_night',
                    'is_dark', 'is_adverse_surface', 'is_adverse_weather',
                    'is_high_speed', 'is_rural', 'is_single_carriageway']

for feat in features_created:
    pct = df[feat].mean() * 100
    print(f"  {feat:<25} {df[feat].sum():>8,}  ({pct:.1f}%)")

Target distribution:
  Fatal or Serious (1): 161,400 (25.2%)
  Slight (0):           479,049 (74.8%)

Features created successfully.

Feature summary:
  target                     161,400  (25.2%)
  is_rush_hour               277,409  (43.3%)
  is_weekend                 168,142  (26.3%)
  is_night                    71,635  (11.2%)
  is_dark                    186,498  (29.1%)
  is_adverse_surface         173,993  (27.2%)
  is_adverse_weather          94,754  (14.8%)
  is_high_speed              131,608  (20.5%)
  is_rural                   231,040  (36.1%)
  is_single_carriageway      463,101  (72.3%)


### Why We Create New Features

The raw data contains coded numbers like light_conditions = 4 or 
road_surface_conditions = 2. Feature engineering converts these into 
clear signals the model can learn from.

Each new feature is binary — 1 means yes, 0 means no:

- **is_rush_hour** — 7-9am or 4-7pm. 43.3% of casualties happen here
- **is_weekend** — different driving patterns to weekdays (26.3%)
- **is_night** — midnight to 5am. Few accidents but more severe (11.2%)
- **is_dark** — any darkness including streetlit roads at night (29.1%)
- **is_adverse_surface** — wet, snow, ice, flood, oil or mud (27.2%)
- **is_adverse_weather** — rain, snow, fog or high winds (14.8%)
- **is_high_speed** — 60mph or above, highest fatal/serious rate (20.5%)
- **is_rural** — rural roads are deadliest when accidents happen (36.1%)
- **is_single_carriageway** — two-way traffic, no central barrier (72.3%)

In [3]:
# ── DEFINE FEATURES AND TARGET ────────────────────────────────────────────

feature_cols = [
    # Time features
    'hour', 'month', 'is_rush_hour', 'is_weekend', 'is_night',
    # Condition features
    'is_dark', 'is_adverse_surface', 'is_adverse_weather',
    # Road features
    'speed_limit_clean', 'is_high_speed', 'is_rural',
    'is_single_carriageway', 'number_of_vehicles'
]

target_col = 'target'

# Drop rows with any missing values in features
df_model = df[feature_cols + [target_col, 'collision_year']].dropna().copy()

print(f"Rows before dropping missing: {len(df):,}")
print(f"Rows after dropping missing:  {len(df_model):,}")
print(f"Rows dropped:                 {len(df) - len(df_model):,}")

# ── TRAIN / TEST SPLIT ────────────────────────────────────────────────────
# Train on 2020-2022, test on 2023-2024
train = df_model[df_model['collision_year'] <= 2022]
test  = df_model[df_model['collision_year'] > 2022]

X_train = train[feature_cols]
y_train = train[target_col]
X_test  = test[feature_cols]
y_test  = test[target_col]

print(f"\nTrain set: {len(X_train):,} rows (2020-2022)")
print(f"Test set:  {len(X_test):,} rows (2023-2024)")
print(f"\nTarget balance in training set:")
print(f"  Fatal/Serious (1): {y_train.sum():,} ({y_train.mean()*100:.1f}%)")
print(f"  Slight (0):        {(y_train==0).sum():,} ({(1-y_train.mean())*100:.1f}%)")
print(f"\nTarget balance in test set:")
print(f"  Fatal/Serious (1): {y_test.sum():,} ({y_test.mean()*100:.1f}%)")
print(f"  Slight (0):        {(y_test==0).sum():,} ({(1-y_test.mean())*100:.1f}%)")

Rows before dropping missing: 640,449
Rows after dropping missing:  640,449
Rows dropped:                 0

Train set: 379,213 rows (2020-2022)
Test set:  261,236 rows (2023-2024)

Target balance in training set:
  Fatal/Serious (1): 92,188 (24.3%)
  Slight (0):        287,025 (75.7%)

Target balance in test set:
  Fatal/Serious (1): 69,212 (26.5%)
  Slight (0):        192,024 (73.5%)


In [4]:
# ── SAVE MODEL-READY DATASET ──────────────────────────────────────────────

os.makedirs('../data/processed/', exist_ok=True)

# Save train and test separately
train.to_csv('../data/processed/train.csv', index=False)
test.to_csv('../data/processed/test.csv', index=False)

print("=" * 55)
print("NOTEBOOK 03 COMPLETE — MODEL DATASETS SAVED")
print("=" * 55)
print(f"Features:          {len(feature_cols)}")
print(f"Target:            binary (1=Fatal/Serious, 0=Slight)")
print(f"Train rows:        {len(train):,} (2020-2022)")
print(f"Test rows:         {len(test):,} (2023-2024)")
print(f"Train saved to:    data/processed/train.csv")
print(f"Test saved to:     data/processed/test.csv")
print("=" * 55)
print("NEXT: Notebook 04 — Machine Learning Model")
print("=" * 55)

NOTEBOOK 03 COMPLETE — MODEL DATASETS SAVED
Features:          13
Target:            binary (1=Fatal/Serious, 0=Slight)
Train rows:        379,213 (2020-2022)
Test rows:         261,236 (2023-2024)
Train saved to:    data/processed/train.csv
Test saved to:     data/processed/test.csv
NEXT: Notebook 04 — Machine Learning Model
